In [ ]:
import pandas as pd
import numpy as np
import re

# 1. 데이터 불러오기

In [ ]:
df = pd.read_csv("./data/저출산 소개팅_설문조사_시나리오추가.csv")

In [ ]:
df.head()

In [ ]:
df.info()

# 2. 데이터 전처리

In [ ]:
df_raw = df.copy()

## (1) 컬럼 정리

In [ ]:
# 타임스탬프, 유저이름 - 제거

# ------- 시나리오 기반 설문 문항 --------
#   5개 축(S/F, A/H, E/L, B/T, P/G)에 연결
#   1이면 왼쪽, 5면 오른쪽 규칙을 코드로 고정
#   유저별 타입문자열(value_type)만들기 -> 점수 벡터화(axis_vector)
# -------------------------------------

# ---------- 기타 추천 보완 문항 -----------
#   자녀 계획 및 가족구성, 경제적 지원 및 가사분담, 자녀 가치관  
#   컬럼명 정리 및 원핫인코딩
# -----------------------------------------

# 가중치 문항값을 더해서 유사도, 보완도 추천에 사용

In [ ]:
# 1) 컬럼명 정규화 함수 (공백/따옴표/줄바꿈 정리)
def clean_colname(col: str) -> str:
    col = str(col).strip()  # 앞뒤 공백 제거
    col = col.replace("\n", " ").replace("\t", " ") # 줄바꿈/탭 등을 공백으로 변경 
    col = re.sub(r"\s+", " ", col)      # 공백 여러 개 -> 하나
    col = col.replace('"', "")          # 큰따옴표 제거
    return col

In [ ]:
# 2) 정규화 적용 + 중복 컬럼명 방지(매우 중요)
cleaned_cols = [clean_colname(c) for c in df.columns]

In [ ]:
# 정규화 후 같은 이름이 중복되면 모델링/전처리에서 에러가 나므로, 자동으로 _1, _2 같은 접미사를 붙여 유니크하게 만든다.
def make_unique(names):
    seen = {}
    out = []
    for n in names:
        if n not in seen:
            seen[n] = 0
            out.append(n)
        else:
            seen[n] += 1
            out.append(f"{n}_{seen[n]}")
    return out

df.columns = make_unique(cleaned_cols)

In [ ]:
# 3) Unnamed 같은 잔재 컬럼 제거
unnamed_cols = [c for c in df.columns if str(c).lower().startswith("unnamed")]
if unnamed_cols:
    df = df.drop(columns=unnamed_cols)

In [ ]:

# 4) 짧은 컬럼명으로 rename (핵심)
rename_map = {
    "타임스탬프": "timestamp",
    "0. 당신의 성함": "user_name",
    "0. 당신의 이상형": "ideal_type",

    "1-1. 희망하는 자녀 수": "p_children_count",
    "1-2. 희망하는 자녀 구성": "p_children_composition",
    "1-3. 자녀 갖고 싶은 시기": "p_children_timing",
    "1-4. 생물학적 출산이 어려움 발생 시 대안": "p_infertility_alternative",

    # 따옴표 제거(clean_colname) 후의 이름을 기준으로 매칭됨
    "1. 자녀 계획 및 가족 구성 항목에 대해 중요도": "importance_family_plan",

    # 시나리오(정수형) 문항들
    '아이가 오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?': "sc_toothbrushing",
    "평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다.": "sc_bedtime_story",
    "경쟁 상황에서의 태도 아이가 운동 경기나 대회에서 아쉽게 2등을 했습니다. 아이는 충분히 잘했다고 기뻐하는데, 당신의 마음속 생각은?": "sc_competition_2nd",
    "재능 발견과 교육 아이가 특정 분야(예: 피아노, 운동)에 천재적인 재능을 보입니다. 이때 당신의 교육 방향은?": "sc_talent_education",
    "두 사람의 훈육 방식이 부딪힐 때, 누구의 의견을 따라야 한다고 생각하시나요?": "sc_discipline_conflict",
    "한 명은 퇴근 후 아이와 놀아주고, 한 명은 밀린 집안일을 해야 하는 상황입니다.": "sc_play_vs_chores",
    "맞벌이 상황 등에서 조부모님이 아이를 봐주겠다고 제안하신다면?": "sc_grandparents_help",
    # 따옴표는 clean_colname에서 제거되므로, 예시 문장 안 따옴표도 없는 버전으로 매칭
    "양가 어르신들이 본인의 가치관과 다른 육아 조언(예: 애를 너무 손타게 키운다, 사탕 좀 주면 어떠냐)을 하실 때 당신의 생각은?": "sc_inlaws_advice",
    "주말에 아이와 동물원에 가기로 했는데, 아침에 일어나니 갑자기 비가 옵니다. 이때 당신의 반응은?": "sc_rainy_zoo",
    "아이의 교육 자금이나 미래 리스크를 대비하는 당신의 생각은?": "sc_education_fund_risk",

    "4-1. 자녀 교육비/양육비 지출 비중": "e_spending_ratio",
    "4-2. 육아 휴직, 양육 부담": "e_parental_leave_burden",
    "4. 경제적 지원 및 가사 분담에 대해 중요도": "importance_econ_support",

    "5-1. 자녀 가치관, 어떤 사람이 되길 바라는가?": "child_values_open",
    "5. 자녀 가치관에 대해 중요도": "importance_child_values",
}

In [ ]:
# rename 시도하기 전에 "매칭이 안 되는 컬럼"이 있는지 점검
missing_in_df = [k for k in rename_map.keys() if k not in df.columns]
if missing_in_df:
    print("\n[주의] rename_map에 있는데 df.columns에서 못 찾은 원문 컬럼명들이 있어요.")
    print("-> 보통 컬럼명에 공백/특수문자 차이로 생깁니다. 아래 목록 확인:")
    for m in missing_in_df:
        print(" -", m)

In [ ]:
# 실제 rename 적용 (존재하는 것만 rename)
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

In [ ]:
# 5) 타입 정리 (최소한으로만)
# timestamp는 추천 시스템에서 시계열을 쓰지 않아도, 기록/정렬에 유용하므로 datetime으로 바꿔둔다.
if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

In [ ]:
# 6) 컬럼 그룹 만들기 (다음 단계 전처리/벡터화에서 매우 유용)
META_COLS = [c for c in ["timestamp", "user_name", "ideal_type"] if c in df.columns]

PLAN_COLS = [c for c in [
    "p_children_count",
    "p_children_composition",
    "p_children_timing",
    "p_infertility_alternative"
] if c in df.columns]

IMPORTANCE_COLS = [c for c in [
    "importance_family_plan",
    "importance_econ_support",
    "importance_child_values"
] if c in df.columns]

ECON_COLS = [c for c in [
    "e_spending_ratio",
    "e_parental_leave_burden"
] if c in df.columns]

TEXT_COLS = [c for c in ["child_values_open"] if c in df.columns]

SCENARIO_COLS = [c for c in df.columns if c.startswith("sc_")]

print("\n==== 컬럼 그룹 요약 ====")
print("META:", META_COLS)
print("PLAN:", PLAN_COLS)
print("IMPORTANCE:", IMPORTANCE_COLS)
print("ECON:", ECON_COLS)
print("TEXT:", TEXT_COLS)
print("SCENARIO:", SCENARIO_COLS)

In [ ]:
print("\n==== 최종 df 요약 ====")
print("shape:", df.shape)
print("dtypes:\n", df.dtypes)

In [ ]:
# 시나리오 값이 정말 정수/범위가 합리적인지 빠르게 확인
if SCENARIO_COLS:
    print("\n==== 시나리오 응답 값 분포(기본) ====")
    print(df[SCENARIO_COLS].describe())

## (2) mbti형 컬럼 생성

In [ ]:
# --------------- 문제 발생 -----------------
# 1~5값으로 데이터를 받으면 중립값이 생겨서 정확히 mbti형으로 표현하기 힘들다.
# 중립값이 없는 형식으로 받았어야 함
# 타이브레이크를 거는 형식으로 구성해서 강제로 중립값을 없앤다. 

In [ ]:
scenario_cols = [c for c in df.columns if c.startswith("sc_")]

In [ ]:
print("찾은 scenario_ 컬럼 개수:", len(scenario_cols))
scenario_cols

In [ ]:
# 1) "두 개씩" 묶기: (0,1), (2,3), (4,5), (6,7), (8,9)
pairs = list(zip(scenario_cols[0::2], scenario_cols[1::2]))

In [ ]:
# 2) 5개 축 정의 (왼쪽 글자, 오른쪽 글자, 축 이름 key)
#    - key는 새로 만들 컬럼명에 사용
axes = [
    ("S", "F", "SF_parenting_execution"),  # 양육 실행 스타일
    ("A", "H", "AH_education_growth"),  # 교육/성장 우선순위
    ("E", "L", "EL_co_parenting"),  # 공동양육 운영 방식
    ("B", "T", "BT_family_boundary"),  # 확장가족/경계
    ("P", "G", "PG_planning_risk"),  # 계획/리스크 대응
]

In [ ]:
# 축 개수와 pair 개수가 맞는지 확인
if len(pairs) != len(axes):
    raise ValueError(f"pairs({len(pairs)})와 axes({len(axes)}) 개수가 맞지 않습니다.")

In [ ]:
# 3) 타이브레이크 함수: "무조건 한 쪽 글자"를 반환하는 함수
# v1, v2: 1~5 응답값(결측이면 이미 3으로 채운 뒤 들어오게 처리)
# left/right: 축의 왼쪽/오른쪽 글자
# - signed 변환: (값 - 3)
# - 합계가 양수면 right, 음수면 left
# - 합계가 0이면(중립 가능) 타이브레이크로 중립 제거

def pick_letter_no_neutral(v1: float, v2: float, left: str, right: str) -> str:

    s1 = v1 - 3
    s2 = v2 - 3
    total = s1 + s2

    if total > 0:
        return right
    if total < 0:
        return left

    # ---- 여기부터가 "중립 방지" 타이브레이크 ----
    # 1) 두 번째 문항이 3이 아니면, 그 문항 방향으로 결정
    if s2 > 0:
        return right
    if s2 < 0:
        return left

    # 2) 첫 번째 문항이 3이 아니면, 그 문항 방향으로 결정
    if s1 > 0:
        return right
    if s1 < 0:
        return left

    # 3) 둘 다 정확히 3이면 완전 중립이므로 기본값으로 한쪽 고정
    #    기본값은 오른쪽으로 두었습니다(원하면 왼쪽으로 바꿔도 됨).
    return right


In [ ]:
# 4) 축별 점수/글자 컬럼 생성
#    - axis_*_mean : 두 문항 평균(1~5 범위, float)
#    - axis_*_sum  : 두 문항 합(2~10 범위, int)
#    - axis_*_letter : 최종 글자(중립 없음)

for (left, right, axis_key), (col1, col2) in zip(axes, pairs):

    # ---- 4-1) 값이 숫자형이 아닐 수 있으니 안전하게 숫자로 변환 ----
    # errors="coerce" : 숫자로 못 바꾸면 NaN(결측)로 만든다.
    v1 = pd.to_numeric(df[col1], errors="coerce")
    v2 = pd.to_numeric(df[col2], errors="coerce")

    # ---- 4-2) 결측치 처리 ----
    # 결측치는 "중립(3)"으로 채워서 파이프라인이 깨지지 않게 한다.
    # (타입은 타이브레이크 때문에 결국 한쪽 글자로 결정됨)
    v1 = v1.fillna(3)
    v2 = v2.fillna(3)

    # ---- 4-3) 축 점수 만들기 (분석/디버깅용으로 같이 저장) ----
    df[f"{axis_key}_sum"] = (v1 + v2).astype(int)      # 2~10
    df[f"{axis_key}_mean"] = (v1 + v2) / 2.0           # 1~5 (float)

    # ---- 4-4) 축 글자 결정(중립 없음) ----
    # row-wise로 v1, v2를 넣어 pick_letter_no_neutral 실행
    df[f"{axis_key}_letter"] = [
        pick_letter_no_neutral(a, b, left, right)
        for a, b in zip(v1.tolist(), v2.tolist())
    ]

    # ---- 4-5) 어떤 시나리오가 어떤 축에 들어갔는지 로그로 확인 ----
    print(f"\n[{axis_key}] {left} vs {right}")
    print(" - 사용 문항 1:", col1)
    print(" - 사용 문항 2:", col2)

In [ ]:
# 5) 최종 "MBTI형(5축)" 타입 문자열 만들기
letter_cols = [f"{axis_key}_letter" for _, _, axis_key in axes]
df["childcare_type_5axis"] = df[letter_cols].astype(str).agg("".join, axis=1)

In [ ]:
print("\n==== 생성된 타입 분포(빈도) ====")
print(df["childcare_type_5axis"].value_counts())

print("\n==== 생성된 컬럼 미리보기 ====")
preview_cols = (
    scenario_cols
    + [f"{k}_sum" for _, _, k in axes]
    + [f"{k}_mean" for _, _, k in axes]
    + letter_cols
    + ["childcare_type_5axis"]
)
print(df[preview_cols].head())

In [ ]:
df.info()

In [ ]:
df.head()

## (3) 스케일링

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, normalize
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
print("\n==== 컬럼 그룹 요약 ====")
print("META:", META_COLS)
print("PLAN:", PLAN_COLS)
print("IMPORTANCE:", IMPORTANCE_COLS)
print("ECON:", ECON_COLS)
print("TEXT:", TEXT_COLS)
print("SCENARIO:", SCENARIO_COLS)